# CSV Order Analyzer: A Book-Style Walkthrough

This notebook is written like a small textbook. Each chapter explains one part of the project, then shows code, then explains the result.

Run the notebook top-to-bottom with the project `.venv` kernel selected.

## 1. What problem does this project solve?

CSV files are often messy. Rows can be missing values, fields can be typed incorrectly, and duplicate records can appear without warning. This project turns a raw order CSV into a cleaned dataset plus a summary report.

At a high level, the project answers three questions:
- Which rows are valid?
- Which rows are invalid or duplicated?
- What useful totals can we compute from the valid rows?

## 2. What files matter most?

The project is split into small files so each part has a clear job.

- `src/main.py`: command-line entrypoint; reads arguments and runs the pipeline.
- `src/processing.py`: cleans rows, validates fields, and detects duplicates.
- `src/summary.py`: computes totals and formats a human-readable report.
- `src/io.py`: writes CSV output files consistently.
- `notebooks/demo.ipynb`: shorter demo notebook with charts.
- `streamlit_app.py`: interactive web demo.
- `reports/`: output folder for cleaned CSVs and summary text.

In [ ]:
# Chapter 2 helper: show the repository's top-level files
from pathlib import Path
repo_root = Path('..').resolve()
for item in sorted(repo_root.iterdir()):
    print(item.name)

## 3. Before anything else: make sure the environment is correct

This notebook expects the project virtual environment to be selected. If the kernel is wrong, imports like `src.processing` may fail even when the code is correct.

If you see an import error, the fix is usually the interpreter or notebook kernel, not the project logic.

In [ ]:
# Environment check
import sys
print('Python executable:', sys.executable)

try:
    import pandas as pd
    import matplotlib
    print('pandas version:', pd.__version__)
    print('matplotlib version:', matplotlib.__version__)
except Exception as error:
    print('Import warning:', error)

## 4. The command-line entrypoint: `src/main.py`

This file is the orchestrator. It does not contain the validation logic itself. Instead, it connects the smaller modules together.

The flow is:
1. Parse command-line arguments.
2. Read the CSV file into rows.
3. Send rows into the processing function.
4. Summarize the results.
5. Save outputs into `reports/`.

In [ ]:
# Read the same CSV that the CLI uses
import csv
from pprint import pprint
from src.processing import classify_orders
from src.summary import summarize_orders, format_report

sample_path = Path('..') / 'data' / 'sample_orders.csv'
with sample_path.open(newline='', encoding='utf-8') as file:
    rows = list(csv.DictReader(file))

print('Rows loaded from CSV:', len(rows))
valid_orders, invalid_orders, duplicate_orders = classify_orders(rows)
print('Valid rows:', len(valid_orders))
print('Invalid rows:', len(invalid_orders))
print('Duplicate rows:', len(duplicate_orders))

try:
    import pandas as pd
    display(pd.DataFrame(valid_orders).head())
except Exception:
    pprint(valid_orders[:3])

## 5. What `src/processing.py` does

This is the core logic of the project. It takes rows from the CSV and decides what happens to each one.

For each row, the processing logic usually does this:
- normalize text fields so they are consistent,
- check that required fields exist and are not blank,
- check that numbers can be parsed and are valid,
- compute row-level revenue when the row is valid,
- check whether the order is a duplicate.

The result is three groups: valid rows, invalid rows, and duplicate rows.

In [ ]:
# Look at one valid row in detail
if valid_orders:
    first_valid = valid_orders[0]
    for key, value in first_valid.items():
        print(f'{key}: {value}')
else:
    print('No valid rows were returned.')

## 6. What happens to invalid rows?

Invalid rows are rows that fail one or more checks. Instead of silently dropping them, the project writes them to a separate file and stores the reason in an `error` column.

This makes the cleaning process transparent. You can always inspect what went wrong.

In [ ]:
# Inspect invalid rows
if invalid_orders:
    try:
        import pandas as pd
        invalid_df = pd.DataFrame(invalid_orders)
        display(invalid_df.head())
    except Exception:
        pprint(invalid_orders[:3])
else:
    print('No invalid rows were found in this sample CSV.')

## 7. What happens to duplicates?

Duplicates are rows that match a key the project treats as unique. In this project, the first occurrence is kept and later matches are written to the duplicate output file.

That design choice keeps the behavior simple and predictable. It also lets you explain the rule clearly in an interview or presentation.

In [ ]:
# Inspect duplicate rows
if duplicate_orders:
    try:
        import pandas as pd
        duplicate_df = pd.DataFrame(duplicate_orders)
        display(duplicate_df.head())
    except Exception:
        pprint(duplicate_orders[:3])
else:
    print('No duplicate rows were found in this sample CSV.')

## 8. What `src/summary.py` does

Once the project has a clean set of valid rows, it can summarize the data. This module answers questions like:
- What is the total revenue?
- Which product earned the most?
- Which customer spent the most?
- What is the average order value?

The summary module is separate from the processing module on purpose. That separation keeps the code easier to test and easier to explain.

In [ ]:
# Summarize the processed rows
summary = summarize_orders(valid_orders, invalid_orders, duplicate_orders)
print(format_report(summary))

### Reading the summary output

The summary text is meant for humans. It gives a concise explanation of what the project found. In practice, this is the kind of output you would show in a terminal or include in a report file.

## 9. What `src/io.py` does

The I/O module is the file-writing layer. It takes the cleaned rows, invalid rows, and duplicate rows and writes them to CSV files in a consistent format.

This is useful because the rest of the project can think in Python objects, while this module handles the details of writing files on disk.

## 10. What the CLI writes to disk

After the pipeline finishes, the project writes these files into `reports/`:
- `clean_orders.csv`
- `invalid_orders.csv`
- `duplicate_orders.csv`
- `summary.txt`

The notebook demo also writes `demo.html` and screenshot images for sharing.

In [ ]:
# Show the summary file that the CLI wrote
summary_file = Path('..') / 'reports' / 'summary.txt'
if summary_file.exists():
    print(summary_file.read_text(encoding='utf-8'))
else:
    print('summary.txt not found at', summary_file)

## 11. How the Streamlit app fits in

The Streamlit app uses the same processing and summary functions, but wraps them in a small web interface. That means the logic is reused; only the interface changes.

This is a good example of separation of concerns: one set of functions does the work, and multiple interfaces can reuse that work.

## 12. How to use this notebook as a learner

A strong way to learn with this notebook is to do three passes:
1. Read the explanation without running anything.
2. Run the code cells and inspect the results.
3. Modify one small thing and predict the outcome before running again.

That third step is where a lot of real understanding happens.

## 13. Suggested exercises

1. Add a duplicate row to `data/sample_orders.csv` and run the notebook again.
2. Remove a required field from one row and see how the invalid report changes.
3. Ask ChatGPT to explain one function in `src/processing.py` line by line.
4. Change a validation rule and compare the new output with the old output.

These exercises turn the notebook into an interactive lesson rather than a static file.

## 14. Final takeaway

This project is small enough to understand, but complete enough to show real software engineering skills. It combines reading files, validating data, handling errors, writing reports, testing logic, and presenting results clearly.

If you can explain the notebook in your own words after running it, you understand the project.

## 15. Packaging, tests, CI, and deployment

### Packaging

Install locally for development:
```bash
pip install -e .
```

### Running tests

Use the lightweight runner during development:
```bash
python run_tests.py
# or
pytest
```

### CI

The repository includes a GitHub Actions workflow at `.github/workflows/ci.yml` that runs tests and basic checks.

### Streamlit deployment

Run locally with:
```bash
streamlit run streamlit_app.py
```

For deployment, Streamlit Cloud or a simple Docker container are good options; include a minimal `requirements.txt` and point the host to `streamlit_app.py`.

In [ ]:
# Helpful commands to copy-paste (prints for convenience)
commands = [
    'pip install -e .',
    'python run_tests.py',
    'pytest',
    'python -m src.main --input data/sample_orders.csv --reports-dir reports',
    'streamlit run streamlit_app.py'
]
for c in commands:
    print(c)

## 16. Adding a license and README badges

- Add an `LICENSE` file (MIT recommended for portfolio projects).
- Add README badges for build status, Python version, and PyPI (if published). Example badges are available at https://shields.io.
- Update `PORTFOLIO_WRITEUP.md` with links to the executed notebooks and screenshots in `reports/`.